In [2]:
import os
import json
import pandas as pd

base_path = ""
ir_base = os.path.join(base_path, "single_channel_ir_1")
meta_base = os.path.join(base_path, "metadata")

rows = []

# Walk through room types (Apartments, Auditorium, etc.)
for room_type in os.listdir(ir_base):
    room_type_path = os.path.join(ir_base, room_type)
    if not os.path.isdir(room_type_path):
        continue
    
    # Walk through each room index (Apartments_idx_0, etc.)
    for room_id in os.listdir(room_type_path):
        room_path = os.path.join(room_type_path, room_id)
        if not os.path.isdir(room_path):
            continue

        # Walk through each wav file
        for file in os.listdir(room_path):
            if not file.endswith(".wav"):
                continue

            # S000_R000_hybrid_IR.wav -> S000_R000
            stem = file.replace("_hybrid_IR.wav", "")

            rir_path = os.path.join(room_path, file)
            json_path = os.path.join(meta_base, room_type, room_id, f"{stem}.json")

            if not os.path.exists(json_path):
                continue

            with open(json_path, "r") as f:
                meta = json.load(f)
            
            
            rows.append({
                "room_type": room_type,
                "room_id": room_id,
                "stem": stem,
                "rir_path": rir_path,
                "src_x": meta["src_loc"][0],
                "src_y": meta["src_loc"][1],
                "src_z": meta["src_loc"][2],
                "rec_x": meta["rec_loc"][0],
                "rec_y": meta["rec_loc"][1],
                "rec_z": meta["rec_loc"][2],
            })

df = pd.DataFrame(rows)
print(df.shape)
print(df.head())



(0, 0)
Empty DataFrame
Columns: []
Index: []


In [8]:
print(f"Total unique room_ids: {df['room_id'].nunique()}")
print(f"\nPer room_type:")
print(df.groupby('room_type')['room_id'].nunique())

Total unique room_ids: 260

Per room_type:
room_type
Apartments                58
Auditorium                 2
Bathrooms                 28
Bedrooms                  34
Cafe                       2
ListeningRoom              4
LivingRoomsWithHallway    40
MeetingRoom               36
Office                    16
Restaurants               40
Name: room_id, dtype: int64


In [10]:
import numpy as np

# Get unique rooms
rooms = df["room_id"].unique()
rooms = np.array(rooms)
np.random.seed(42)
np.random.shuffle(rooms)

# 80/10/10 split
n = len(rooms)
train_rooms = rooms[:int(n * 0.8)]
val_rooms = rooms[int(n * 0.8):int(n * 0.9)]
test_rooms = rooms[int(n * 0.9):]

train_df = df[df["room_id"].isin(train_rooms)].reset_index(drop=True)
val_df = df[df["room_id"].isin(val_rooms)].reset_index(drop=True)
test_df = df[df["room_id"].isin(test_rooms)].reset_index(drop=True)

print(f"Rooms  — train: {len(train_rooms)}, val: {len(val_rooms)}, test: {len(test_rooms)}")
print(f"Samples — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

Rooms  — train: 208, val: 26, test: 26
Samples — train: 227932, val: 20650, test: 27702
